In [2]:
# 09_ablation_tables.ipynb
# Aggregate the multi-seed ablation (notebook 08) into mean +/- 95% CI tables
# for the three ablations (components / threshold alpha / MMD stage), per classifier
# and overall. Output: ../results/tables/ablation_*.csv   ENV: pandas, numpy, scipy.
import os
import numpy as np
import pandas as pd
from scipy import stats

TABLES_DIR = "../results/tables"
raw = pd.read_csv(os.path.join(TABLES_DIR, "raw_ablation.csv"), keep_default_na=False)
METRICS = ["AUC", "F1", "G_mean", "PR_AUC"]
for m in METRICS:
    raw[m] = pd.to_numeric(raw[m], errors="coerce")

def ci95(v):
    v = np.asarray(v, float); n = len(v)
    if n <= 1: return (round(float(v.mean()), 4) if n else np.nan), 0.0
    return round(float(v.mean()), 4), round(float(stats.t.ppf(0.975, n-1)*v.std(ddof=1)/np.sqrt(n)), 4)

# per (config, classifier): mean over 16 datasets' seed-means, with CI over the per-dataset means
def summarise(group):
    sub = raw[raw["config_group"] == group]
    out = []
    for cfg in sub["config"].unique():
        for clf in ["RF", "LGBM", "MLP"]:
            s = sub[(sub["config"] == cfg) & (sub["classifier"] == clf)]
            # average over seeds within each dataset first, then over datasets
            per_ds = s.groupby("dataset")[METRICS].mean()
            rec = {"config": cfg, "classifier": clf, "n_datasets": per_ds.shape[0]}
            for m in METRICS:
                mu, h = ci95(per_ds[m].values)
                rec[f"{m}_mean"] = mu; rec[f"{m}_ci95"] = h
            out.append(rec)
    df = pd.DataFrame(out)
    df.to_csv(os.path.join(TABLES_DIR, f"ablation_{group}.csv"), index=False)
    # compact overall (mean over the three classifiers) for the main-text table
    comp = (df.groupby("config")[[f"{m}_mean" for m in METRICS]].mean().round(4))
    print(f"\n===== {group} (overall mean over classifiers) =====")
    print(comp.to_string())
    return df

for g in ["component", "alpha", "mmd"]:
    summarise(g)
print("\nDONE. ablation_component.csv / ablation_alpha.csv / ablation_mmd.csv in", TABLES_DIR)


===== component (overall mean over classifiers) =====
              AUC_mean  F1_mean  G_mean_mean  PR_AUC_mean
config                                                   
comp_density    0.8787   0.7213       0.6151       0.5484
comp_full       0.8799   0.7236       0.6150       0.5497
comp_lof        0.8790   0.7233       0.6130       0.5474
comp_sep        0.8810   0.7238       0.6142       0.5490

===== alpha (overall mean over classifiers) =====
                 AUC_mean  F1_mean  G_mean_mean  PR_AUC_mean
config                                                      
alpha_adaptive     0.8803   0.7239       0.6181       0.5487
alpha_fixed_0.1    0.8799   0.7227       0.6159       0.5468
alpha_fixed_0.2    0.8799   0.7236       0.6150       0.5497
alpha_fixed_0.3    0.8809   0.7241       0.6171       0.5499

===== mmd (overall mean over classifiers) =====
         AUC_mean  F1_mean  G_mean_mean  PR_AUC_mean
config                                              
mmd_off    0.8802   0.724